# Pendahuluan

Di dalam material ini, kamu akan mempelajari Data Pima Indians Diabetes dan melakukan elaborasi data (*data understanding*, *data quality*, EDA, hingga *feature selection*). Jalankan setiap sel secara berurutan dari atas ke bawah.

## Introduction to Capstone Project

Diabetes melitus merupakan salah satu penyakit tidak menular dengan beban terbesar di dunia. Menurut IDF Diabetes Atlas edisi ke-11 (2025), sekitar 589 juta orang dewasa usia 20–79 tahun (11,1%, atau 1 dari 9 orang dewasa) hidup dengan diabetes, dan sekitar 252 juta di antaranya belum terdiagnosis. Diabetes menyebabkan lebih dari 3,4 juta kematian per tahun, dan jumlah penderitanya diproyeksikan mencapai 853 juta pada 2050. Karena banyak kasus yang tidak terdeteksi, skrining dini pada kelompok berisiko menjadi penting agar komplikasi (penyakit jantung, gagal ginjal, kerusakan saraf, dan kebutaan) dapat dicegah atau ditunda.

Tujuan bisnis yang ingin dicapai adalah membangun model prediksi diabetes berdasarkan hasil pemeriksaan klinis sederhana untuk membantu tenaga kesehatan melakukan skrining awal secara cepat dan akurat.

Sumber: International Diabetes Federation, *IDF Diabetes Atlas 11th Edition* (2025), https://diabetesatlas.org/

**Problem Statement**

Masalah yang ingin diselesaikan adalah mendeteksi pasien yang menderita diabetes secara tepat dan akurat berdasarkan hasil pemeriksaan klinis. Untuk itu perlu dianalisis faktor-faktor klinis yang berhubungan dengan status diabetes pasien.

**About Datasets**

Dataset yang digunakan adalah **Pima Indians Diabetes Database** dari Kaggle: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database

Data ini berasal dari National Institute of Diabetes and Digestive and Kidney Diseases (NIDDK). Seluruh pasien adalah perempuan keturunan Pima Indian berusia minimal 21 tahun. Dataset terdiri dari 768 baris, 8 atribut, dan 1 target. Karena Kaggle memerlukan login untuk mengunduh, notebook ini memuat salinan publik yang identik dari repositori GitHub (lihat sel pemuatan data). Kamu juga bisa mengunduh `diabetes.csv` dari Kaggle lalu mengunggahnya ke Colab.

Kolom pada dataset:

1. Pregnancies: jumlah kehamilan yang pernah dialami pasien.
2. Glucose: konsentrasi glukosa plasma 2 jam setelah tes toleransi glukosa oral (mg/dl).
3. BloodPressure: tekanan darah diastolik (mmHg).
4. SkinThickness: ketebalan lipatan kulit trisep (mm).
5. Insulin: kadar insulin serum 2 jam (mu U/ml).
6. BMI: indeks massa tubuh, berat (kg) dibagi tinggi (m) kuadrat.
7. DiabetesPedigreeFunction: skor riwayat diabetes dalam keluarga (faktor genetik).
8. Age: usia pasien (tahun).
9. Outcome: 0 = tidak diabetes, 1 = diabetes.

**Objective of Capstone 1**

Melakukan data preprocessing termasuk Exploratory Data Analysis untuk menggali insight dari data pasien diabetes, sampai proses feature selection dan dimensionality reduction. Hasil akhirnya adalah insight tentang data pasien diabetes dan data yang siap dimodelkan pada tahap selanjutnya.

**Metrics**

Metrics yang digunakan untuk memantau solusi:
1. Jumlah penderita diabetes dan proporsinya terhadap seluruh pasien.
2. Rata-rata kadar glukosa, BMI, dan usia pasien diabetes dibandingkan pasien non-diabetes.
3. Korelasi status diabetes terhadap tiap variabel klinis (glukosa, BMI, usia, jumlah kehamilan, tekanan darah, riwayat keluarga).

**Persiapan**

Menyiapkan libraries yang diperlukan dan memuat dataset, lalu melihat gambaran umum data.

In [ ]:
#Import libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline

In [ ]:
# Memuat dataset Pima Indians Diabetes (salinan publik dari dataset Kaggle uciml/pima-indians-diabetes-database)
url = "https://raw.githubusercontent.com/plotly/datasets/master/diabetes.csv"
data = pd.read_csv(url)
# Jika memakai file hasil unduhan Kaggle: data = pd.read_csv("diabetes.csv")

# Menampilkan lima baris teratas
data.head()

In [ ]:
# Menampilkan lima baris terbawah
data.tail()

In [ ]:
# Melihat jumlah dataset (jumlah baris dan kolom)
data.shape

In [ ]:
# Melihat columns apa saja yang ada dalam dataset
data.columns

In [ ]:
# Melihat persebaran data
data.describe()

In [ ]:
# Melihat uniqueness data
data.nunique()

In [ ]:
# Melihat informasi dari dataset
data.info()

**Data Quality**

Di sini kita melakukan pengecekan karakter data, nilai yang tidak masuk akal, missing value, duplikat, outlier, dan data imbalance.

In [ ]:
# Kolom kategorikal pada dataset ini hanya target (Outcome)
lst = ['Outcome']
data[lst] = data[lst].astype(object)

# Pelabelan data kategorikal
data['Outcome'] = data['Outcome'].replace({0: 'Non-Diabetes', 1: 'Diabetes'})

# Melihat perubahan
data.dtypes

In [ ]:
# Pisahkan numerical dan categorical
numerical_col = data.select_dtypes(exclude='object')
categorical_col = data.select_dtypes(exclude='number')

# Pengecekan karakter dari data kategorikal
for i in categorical_col:
    print("Berikut ini merupakan informasi: {} dengan {}".format(i, data[i].unique()))

Dari `describe()` terlihat nilai minimum **0** pada kolom Glucose, BloodPressure, SkinThickness, Insulin, dan BMI. Nilai 0 pada kolom-kolom ini tidak mungkin secara medis (orang hidup tidak mungkin memiliki glukosa darah, tekanan darah, atau BMI sama dengan nol), sehingga nilai 0 tersebut sebenarnya adalah **missing value yang dikodekan sebagai 0**. Kolom Pregnancies boleh bernilai 0 karena memang ada pasien yang belum pernah hamil.

In [ ]:
# Menghitung jumlah nilai 0 pada kolom yang tidak boleh bernilai 0
zero_cols = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
(data[zero_cols] == 0).sum()

In [ ]:
# Ganti nilai 0 pada kolom tersebut menjadi NaN
data[zero_cols] = data[zero_cols].replace(0, np.nan)

# Check missing values (jumlah dan persentase)
print("Hasil missing value:\n", data.isnull().sum())
print("\nPersentase missing value (%):\n", (data.isnull().mean() * 100).round(1))

Setelah nilai 0 diubah menjadi NaN, terlihat dua kolom dengan missing value besar: **Insulin (48,7%)** dan **SkinThickness (29,6%)**. Keputusan penanganannya:

1. **Insulin dan SkinThickness dihapus.** Hampir separuh nilai Insulin hilang, sehingga imputasi apa pun akan lebih banyak "mengarang" data daripada mencerminkan kondisi pasien. Imputasi median pada kolom sebesar ini juga menciptakan lonjakan nilai buatan di median, yang membuat rentang IQR menyempit sehingga ratusan data asli terdeteksi sebagai outlier palsu. SkinThickness juga berkorelasi cukup kuat dengan BMI (r ≈ 0,65 pada data yang tidak hilang), jadi informasinya sebagian besar sudah terwakili oleh BMI.
2. **Glucose, BloodPressure, dan BMI diisi dengan median.** Missing value-nya kecil (0,7%–4,6%). Median dipilih karena kolom numerik ini memiliki outlier, sehingga mean akan bias. (Pada data kategorikal kita memakai modus; pada data numerik, median lebih tepat.)

In [ ]:
# Menghapus kolom dengan missing value terlalu besar
data = data.drop(columns=['Insulin', 'SkinThickness'])

# Fillna kolom numerik dengan median (lebih tahan terhadap outlier dibanding mean)
for col in ['Glucose', 'BloodPressure', 'BMI']:
    data[col] = data[col].fillna(data[col].median())

# Check missing values kembali
print("Hasil missing value:\n", data.isnull().sum())

Kita akan melakukan pengecekan rows yang duplikat.

In [ ]:
# Memeriksa duplikat data
data.duplicated().sum()

In [ ]:
# Menghapus data duplikat (jika ada) dan mempertahankan data pertama
data.drop_duplicates(keep='first', inplace=True)

# Memeriksa ulang duplikat data
data.duplicated().sum()

Selanjutnya kita akan menampilkan outliers dan menghapus data yang outlier.

In [ ]:
#Menampilkan boxplot untuk melihat outliers data numerik
data.plot(kind = 'box', subplots = True, layout = (2,4), sharex = False, sharey = False, figsize = (20, 10), color = 'k')
plt.show()

In [ ]:
# Menampilkan data berdasarkan target untuk feature Glucose
sns.boxplot(x='Outcome', y='Glucose', data=data)
plt.show()

Seluruh kolom fitur bersifat numerik kontinu atau diskret: 'Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', dan 'Age'. Outlier dideteksi dengan metode IQR (di luar Q1 − 1,5·IQR dan Q3 + 1,5·IQR).

In [ ]:
#Mencari tahu data yang outliers dengan mendefinisikan fungsi
continous_features = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']
def outliers(data_out, drop = False):
    for each_feature in data_out.columns:
        feature_data = data_out[each_feature]
        Q1 = np.percentile(feature_data, 25.)
        Q3 = np.percentile(feature_data, 75.)
        IQR = Q3-Q1
        outlier_step = IQR * 1.5
        outliers = feature_data[~((feature_data >= Q1 - outlier_step) & (feature_data <= Q3 + outlier_step))].index.tolist()
        if not drop:
            print('For the feature {}, Num of Outliers is {}'.format(each_feature, len(outliers)))
        if drop:
            data.drop(outliers, inplace = True, errors = 'ignore')
            print('Outliers from {} feature removed'.format(each_feature))

# Menampilkan jumlah outlier
outliers(data[continous_features])

In [ ]:
# Menghapus outlier dengan fungsi sebelumnya
outliers(data[continous_features], drop=True)
print("Jumlah data setelah outlier dihapus:", data.shape)

Melihat distribusi beberapa variabel utama.

In [ ]:
# Distribusi usia pasien
data['Age'].hist()
plt.title('Distribusi Usia Pasien')
plt.show()

In [ ]:
# Distribusi kadar glukosa
data['Glucose'].hist()
plt.title('Distribusi Kadar Glukosa')
plt.show()

In [ ]:
# Distribusi BMI
data['BMI'].hist()
plt.title('Distribusi BMI')
plt.show()

Selanjutnya kita akan melakukan pengecekan data, apakah target imbalance?

In [ ]:
# Memeriksa data imbalance
ax = sns.countplot(x='Outcome', data=data)
for label in ax.containers:
    ax.bar_label(label)
sns.despine()
plt.show()
print((data['Outcome'].value_counts(normalize=True) * 100).round(1))

Data **tidak seimbang secara moderat**: sekitar 66,6% pasien Non-Diabetes dan 33,4% pasien Diabetes (rasio sekitar 2:1). Ketidakseimbangan ini belum ekstrem, tetapi berdampak pada pemodelan: model cenderung lebih mudah menebak kelas mayoritas, sehingga **akurasi saja tidak cukup** sebagai ukuran kinerja. Pada Capstone 2 kita akan memakai stratified split, ROC-AUC, recall, dan opsi `class_weight='balanced'`.

**Periksa statistik deskriptif dari kumpulan data (rata-rata, distribusi, dll)**

In [ ]:
# Melihat statistik deskriptif dari data
data.describe()

In [ ]:
# Membandingkan rata-rata tiap variabel antara pasien diabetes dan non-diabetes
data.groupby('Outcome').mean().round(2)

Setelah pembersihan, terdapat 707 pasien dan 6 fitur. Ringkasan statistik deskriptif:

1. Pregnancies: rata-rata 3,77 kali kehamilan, rentang 0–13.
2. Glucose: rata-rata 120,7 mg/dl, rentang 44–198.
3. BloodPressure: rata-rata 72,0 mmHg, rentang 44–104.
4. BMI: rata-rata 32,1, rentang 18,2–50,0. Rata-rata ini sudah masuk kategori obesitas (BMI ≥ 30).
5. DiabetesPedigreeFunction: rata-rata 0,43, rentang 0,078–1,191.
6. Age: rata-rata 32,6 tahun, rentang 21–66 tahun.

Perbandingan per kelompok menunjukkan perbedaan paling mencolok pada **Glucose**: rata-rata 142,0 mg/dl pada pasien Diabetes dibandingkan 110,0 mg/dl pada Non-Diabetes. Pasien Diabetes juga rata-rata lebih tua (37,0 vs 30,5 tahun), memiliki BMI lebih tinggi (34,7 vs 30,8), dan jumlah kehamilan lebih banyak (4,8 vs 3,3). Selisih tekanan darah relatif kecil (74,6 vs 70,6 mmHg).

In [ ]:
# Memvisualisasikan distribusi variabel numerical
numerical_col = data.select_dtypes(exclude='object')
plt.figure(figsize=(16,8))
for index,column in enumerate(numerical_col):
    plt.subplot(2,3,index+1)
    sns.histplot(data=numerical_col,x=column,kde=True)
    plt.xticks(rotation = 90)
plt.tight_layout(pad = 1.0)
plt.show()

In [ ]:
# Memvisualisasikan distribusi variabel numerical berdasarkan target
plt.figure(figsize=(16,8))
for index, column in enumerate(numerical_col):
    plt.subplot(2, 3, index+1)
    sns.kdeplot(data=data, x=column, hue='Outcome', fill=True, common_norm=False, palette='magma')
    plt.xlabel(column.upper(), fontsize=12)
plt.tight_layout(pad = 1.0)
plt.show()

Melihat hubungan antar feature.

In [ ]:
# melihat hubungan usia dan jumlah kehamilan
plt.figure(figsize = (8,6))
sns.regplot(x='Age', y='Pregnancies', data=data, scatter_kws = {'color':'blue'}, line_kws = {'color':'purple'})
plt.title('Sebaran Data Pasien Berdasarkan\nUsia dan Jumlah Kehamilan')
plt.show()

In [ ]:
# melihat hubungan BMI dan kadar glukosa
plt.figure(figsize = (8,6))
sns.regplot(x='BMI', y='Glucose', data=data, scatter_kws = {'color':'blue'}, line_kws = {'color':'purple'})
plt.title('Sebaran Data Pasien Berdasarkan\nBMI dan Kadar Glukosa')
plt.show()

from scipy.stats import pearsonr
print("Age vs Pregnancies :", pearsonr(data['Age'], data['Pregnancies']))
print("BMI vs Glucose     :", pearsonr(data['BMI'], data['Glucose']))

**Kesimpulan dari grafik:**

1. **Age vs Pregnancies**: garis regresi menanjak jelas. Korelasinya positif dan sedang (r ≈ 0,55; p < 0,001). Wajar, karena pasien yang lebih tua punya lebih banyak waktu untuk mengalami kehamilan. Kedua variabel ini sebagian membawa informasi yang sama (usia), sehingga perlu diwaspadai saat interpretasi koefisien regresi logistik.
2. **BMI vs Glucose**: garis regresi sedikit menanjak dengan sebaran titik yang lebar. Korelasinya positif tetapi lemah (r ≈ 0,21; p < 0,001). BMI tinggi cenderung disertai glukosa yang lebih tinggi, tetapi BMI sendiri hanya menjelaskan sebagian kecil variasi glukosa.

Korelasi hanya menunjukkan hubungan linear, bukan hubungan sebab-akibat.

In [ ]:
# Simpan data yang sudah bersih
data.to_csv('data.csv')

# Melabeli kembali target menjadi numerik
df = pd.read_csv('data.csv', index_col=0)
df['Outcome'] = df['Outcome'].replace({'Non-Diabetes': 0, 'Diabetes': 1}).astype(int)

In [ ]:
# melihat korelasi antar variable untuk mencari feature yang penting
plt.figure(figsize=(10,8))
cor = df.corr()
sns.heatmap(cor,annot=True, linewidth=.5, cmap="magma")
plt.title('Korelasi Antar Variable', fontsize = 20)
plt.show()

In [ ]:
cor_matrix = df.corr()
cor_matrix

In [ ]:
#Mengurutkan korelasi terhadap target
cor_matrix['Outcome'].sort_values()

Korelasi Outcome (diabetes) dengan variabel lainnya. Seluruh korelasi bernilai positif, artinya makin tinggi nilai variabel, makin tinggi kecenderungan pasien menderita diabetes.

1. Glucose 0,507 (korelasi positif sedang, paling kuat)
2. BMI 0,283 (korelasi positif lemah)
3. Age 0,278 (korelasi positif lemah)
4. Pregnancies 0,221 (korelasi positif lemah)
5. DiabetesPedigreeFunction 0,174 (korelasi positif sangat lemah)
6. BloodPressure 0,173 (korelasi positif sangat lemah)

(Pedoman umum besaran |r|: < 0,2 sangat lemah, 0,2–0,4 lemah, 0,4–0,6 sedang, 0,6–0,8 kuat, > 0,8 sangat kuat.)

Kesimpulan:

1. Glucose adalah faktor yang paling berhubungan dengan diabetes. Ini sesuai dengan kriteria diagnosis klinis diabetes yang memang berbasis kadar glukosa darah.
2. BMI, Age, dan Pregnancies berhubungan lemah tetapi konsisten dengan diabetes.
3. DiabetesPedigreeFunction dan BloodPressure berkorelasi sangat lemah, tetapi tetap dipertahankan karena secara klinis relevan (faktor keturunan dan hipertensi adalah faktor risiko diabetes), dan model nonlinear seperti Random Forest dapat menangkap hubungan yang tidak terlihat oleh korelasi linear.
4. Tidak ada pasangan fitur yang berkorelasi sangat kuat (tertinggi Age–Pregnancies, r ≈ 0,55), jadi tidak ada fitur yang perlu dibuang karena multikolinearitas berat.

Feature yang dipilih: 'Glucose', 'BMI', 'Age', 'Pregnancies', 'DiabetesPedigreeFunction', dan 'BloodPressure'.

In [ ]:
# Variabel terpilih
variabel = ['Glucose', 'BMI', 'Age', 'Pregnancies', 'DiabetesPedigreeFunction', 'BloodPressure']
analyze = df[variabel]

In [ ]:
# Informasi variabel ke target
fig = plt.figure(figsize=(15,8))
for index in range(len(analyze.columns)):
    plt.subplot(2,3,index + 1)
    sns.boxplot(x=df.Outcome, y=analyze.iloc[:,index])
    plt.xlabel("Outcome", fontsize=12)
    plt.ylabel(analyze.columns[index], fontsize=12)
fig.tight_layout(pad = 1.0)

**Scaling data**

Scaling membuat seluruh variabel numerik memiliki rentang nilai (skala) yang sama sehingga tidak ada satu variabel yang mendominasi variabel lain. Di Capstone 1 scaling hanya dipakai untuk eksplorasi PCA. Pada tahap pemodelan (Capstone 2), scaler di-*fit* hanya pada data training agar tidak terjadi kebocoran data.

In [ ]:
# Mendefinisikan standard scaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df.drop("Outcome", axis=1))

In [ ]:
# Perform PCA for dimensionality reduction
feature_number = len(X_scaled[0])
pca = PCA(n_components=feature_number)
pca.fit(X_scaled)

# Get variance information
variance_ratio = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(variance_ratio)

# Create Scree Plot
plt.plot(range(1, len(variance_ratio) + 1), variance_ratio, marker='o', label='Varians per komponen')
plt.plot(range(1, len(variance_ratio) + 1), cumulative_variance, marker='s', label='Varians kumulatif')
plt.xlabel('Komponen Utama ke-')
plt.ylabel('Proporsi Varians')
plt.title('Scree Plot')
plt.legend()
plt.show()
print("Varians kumulatif:", cumulative_variance.round(3))

In [ ]:
feature_names = df.drop('Outcome', axis=1).columns.to_list()
component_names = [f"PC{i+1}" for i in range(pca.n_components_)]

for component, component_name in zip(pca.components_, component_names):
    feature_indices = np.abs(component).argsort()[::-1]
    retained_features = [feature_names[idx] for idx in feature_indices[:3]]
    print(f"{component_name}: {retained_features}")

Scree plot menunjukkan varians tersebar cukup merata: PC1 hanya menjelaskan sekitar 32,8% varians, dan dibutuhkan 4 komponen untuk mencapai sekitar 82,4% serta 5 komponen untuk sekitar 93,4%. Karena jumlah fitur hanya 6 dan reduksi dimensi tidak menghemat banyak, **PCA tidak dipakai untuk pemodelan**. Model di Capstone 2 tetap memakai 6 fitur asli agar hasilnya mudah diinterpretasikan dan dashboard di Capstone 3 bisa menerima input klinis secara langsung. PCA di sini berfungsi sebagai eksplorasi struktur data.